# Iris Plant Classification: GMDH Sparse Perceptrons vs. Multi-Layer Perceptrons (MLP)

## 1. Problem Overview & Dataset Background

The **Iris Plants Database**, introduced by R. A. Fisher in his classic 1936 paper *"The use of multiple measurements in taxonomic problems"* (*Annual Eugenics*, 7, Part II, 179–188), is one of the best-known benchmarks in pattern recognition.

### Dataset Characteristics (`iris.names`)
- **Total Instances**: 150 samples evenly distributed across **3 classes** (50 instances per class):
  1. **Iris Setosa** (`class_id = 1`)
  2. **Iris Versicolour** (`class_id = 2`)
  3. **Iris Virginica** (`class_id = 3`)
- **Predictive Attributes** (4 continuous measurements in centimeters):
  1. `sepal_len_cm`: Sepal length in cm (range: 4.3 – 7.9, $\mu = 5.84, \sigma = 0.83$, class correlation: $0.7826$)
  2. `sepal_wid_cm`: Sepal width in cm (range: 2.0 – 4.4, $\mu = 3.05, \sigma = 0.43$, class correlation: $-0.4194$)
  3. `petal_len_cm`: Petal length in cm (range: 1.0 – 6.9, $\mu = 3.76, \sigma = 1.76$, class correlation: $0.9490$)
  4. `petal_wid_cm`: Petal width in cm (range: 0.1 – 2.5, $\mu = 1.20, \sigma = 0.76$, class correlation: $0.9565$)
- **Separability**: *Iris Setosa* is linearly separable from the other two species, whereas *Iris Versicolour* and *Iris Virginica* overlap partially and are **not** linearly separable from each other.
- **Data Corrections Verified**: As documented in `iris.names` (identified by Steve Chadwick), two transcription discrepancies in the original UCI archive compared to Fisher's 1936 article have already been corrected in `iris.csv`:
  - **35th sample** (0-based index `34`): `4.9, 3.1, 1.5, 0.2, 1, "Iris Setosa"` (fourth feature corrected to `0.2`).
  - **38th sample** (0-based index `37`): `4.9, 3.6, 1.4, 0.1, 1, "Iris Setosa"` (second and third features corrected to `3.6, 1.4`).

---

## 2. Experiment Objectives

In this notebook, we compare a **growing GMDH network** (with `SoftClip` activations, inter-layer normalization, and simultaneous gradient fine-tuning during constructive growth) against a **Dense Multi-Layer Perceptron (MLP)** of comparable size under two target formulations:
1. **Single-Output Regression Classification**: Predicting the scalar class ID ($y \in \{1, 2, 3\}$) and rounding at inference.
2. **Multi-Output One-Hot Classification**: Predicting one output channel per class ($\mathbf{y} \in \{0, 1\}^3$).

## 3. Imports & Global Configuration

We define our global random number generator (`RNG`), noise level (`STATE_NOISE`), and PyTorch device/dtype settings (`TORCH_DEVICE`, `TORCH_DTYPE`) at the top of the notebook.

In [ ]:
from pathlib import Path
import time

from IPython.display import HTML, clear_output, display
from tabulate import tabulate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from gmdhtorch import (
    GMDHLayer,
    GMDHModel,
    GMDHOutput,
    ModelIncrementer,
    SelectRelativeThreshold,
    SoftClip,
    nrmse,
    resolve_dtype,
)
try:
    from utils import add_noise, make_tensor_dataloader, split_indexes
except ImportError:
    from examples.utils import add_noise, make_tensor_dataloader, split_indexes

# Make Matplotlib figures span 100% of the notebook output width
plt.rcParams["figure.figsize"] = (15, 4)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"
display(
    HTML(
        "<style>"
        ".jp-RenderedImage img, .output_png img, .cell-output-display img { "
        "width: 100% !important; max-width: 100% !important; height: auto !important; }"
        "</style>"
    )
)

# Global configuration constants
SEED: int = 42
RNG: np.random.Generator = np.random.default_rng(SEED)
STATE_NOISE: float = 0.05
TORCH_DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE: torch.dtype = resolve_dtype(None, device=TORCH_DEVICE)

print(f"Device: {TORCH_DEVICE} | Dtype: {TORCH_DTYPE} | Seed: {SEED} | State Noise: {STATE_NOISE}")

## 4. Loading and Inspecting the Iris Dataset

We load `iris.csv` into a Pandas `DataFrame` to inspect the tabular structure and verify the corrected Fisher samples (#35 and #38).

In [ ]:
iris_path = Path("iris.csv") if Path("iris.csv").exists() else Path("examples/iris.csv")
df = pd.read_csv(iris_path)

print(f"Loaded {len(df)} rows from {iris_path}:")
display(df.head(8))

print("\nVerification of Fisher's corrected samples (#35 and #38, 0-based rows 34 and 37):")
display(df.iloc[[34, 37]])

### Extracting Class Mapping

Next, we extract the unique `(class_id, class_name)` pairs from the DataFrame, display the resulting table, and convert it into a lookup dictionary `{class_id: class_name}`.

In [ ]:
classes_df = df[["class_id", "class_name"]].drop_duplicates().reset_index(drop=True)
print("Unique classes in dataset:")
display(classes_df)

classes = dict(classes_df.to_numpy())
print("Class dictionary:", classes)

### Stratified Partitioning into Training, Selection, and Validation Sets (Single-Output)

To ensure balanced representation of all three Iris species across every split, we partition each class's 50 samples with proportions `(3, 1, 2)`:
- **Training set ($3/6 = 50\%$)**: 25 samples per class (75 total) used to fit polynomial candidates and compute gradients.
- **Selection set ($1/6 \approx 16.7\%$)**: 8 samples per class (24 total) used by GMDH to score and select neurons per layer.
- **Validation set ($2/6 \approx 33.3\%$)**: 17 samples per class (51 total) held out strictly for final evaluation.

Here we prepare **single-output scalar targets** ($y \in \{1, 2, 3\}$). Multi-output targets will be constructed in a later section.

In [ ]:
data = df[["sepal_len_cm", "sepal_wid_cm", "petal_len_cm", "petal_wid_cm", "class_id"]].to_numpy()

tr_samples_list, tr_targets_list = [], []
sl_samples_list, sl_targets_list = [], []
vl_samples_list, vl_targets_list = [], []

for k in classes:
    class_data = data[data[:, -1] == k]
    class_samples = class_data[:, :-1]
    class_targets = class_data[:, -1]

    # 3/6: training; 1/6: selection; 2/6: validation
    sel_split, val_split = split_indexes(len(class_samples), (3, 1, 2))

    tr_samples_list.append(class_samples[:sel_split])
    tr_targets_list.append(class_targets[:sel_split])
    sl_samples_list.append(class_samples[sel_split:val_split])
    sl_targets_list.append(class_targets[sel_split:val_split])
    vl_samples_list.append(class_samples[val_split:])
    vl_targets_list.append(class_targets[val_split:])

tr_samples = np.concatenate(tr_samples_list, axis=0)
tr_targets = np.concatenate(tr_targets_list, axis=0)
sl_samples = np.concatenate(sl_samples_list, axis=0)
sl_targets = np.concatenate(sl_targets_list, axis=0)
vl_samples = np.concatenate(vl_samples_list, axis=0)
vl_targets = np.concatenate(vl_targets_list, axis=0)

print(f"Training set shape  : samples={tr_samples.shape}, targets={tr_targets.shape}")
print(f"Selection set shape : samples={sl_samples.shape}, targets={sl_targets.shape}")
print(f"Validation set shape: samples={vl_samples.shape}, targets={vl_targets.shape}")

### Shuffling the Training Set

Because `tr_samples` and `tr_targets` were concatenated class-by-class (`Setosa` first, then `Versicolour`, then `Virginica`), **we must shuffle the training set before performing noise augmentation**. Otherwise, slicing the first third (`tr_samples[:len(tr_samples) // 3]`) would only duplicate samples from Class 1 (*Iris Setosa*).

In [ ]:
tr_shuffle_indexes = RNG.permutation(len(tr_samples))
tr_samples = tr_samples[tr_shuffle_indexes]
tr_targets = tr_targets[tr_shuffle_indexes]

print("First 10 training targets after shuffling:", tr_targets[:10].astype(int))

### Noise Augmentation (Duplicating $1/3$ of the Training Set)

When `STATE_NOISE > 0.0`, we augment the training set by taking one-third of the shuffled training samples (which spans all classes), adding zero-mean Gaussian noise $\mathcal{N}(0, \text{STATE\_NOISE}^2)$ to the input features while keeping their original class labels intact, concatenating them to the training set, and reshuffling.

If `STATE_NOISE == 0.0`, no duplication is performed.

In [ ]:
if STATE_NOISE > 0.0:
    qtd_original_samples = len(tr_samples)
    qtd_noise_samples = len(tr_samples) // 3
    tr_noise_samples = add_noise(tr_samples[:qtd_noise_samples], STATE_NOISE, RNG)
    tr_noise_targets = tr_targets[:qtd_noise_samples].copy()

    unique_dup, counts_dup = np.unique(tr_noise_targets.astype(int), return_counts=True)

    tr_samples = np.concatenate([tr_samples, tr_noise_samples], axis=0)
    tr_targets = np.concatenate([tr_targets, tr_noise_targets], axis=0)

    # Reshuffle augmented training set
    tr_shuffle_indexes = RNG.permutation(len(tr_samples))
    tr_samples = tr_samples[tr_shuffle_indexes]
    tr_targets = tr_targets[tr_shuffle_indexes]

    table = [["Number of original samples", qtd_original_samples], ["Number of noise samples", qtd_noise_samples]]
    t = np.column_stack([unique_dup, counts_dup]).tolist()
    for row in t:
        row[0] = f"Class {row[0]}"
    table.extend(t)
    table.append(["Final training size", len(tr_samples)])
    display(tabulate(table, tablefmt="html"))


## 5. DataLoader & Model Factory Functions

### 5.1 DataLoader Factory
We define a helper function `create_dataloaders` using `make_tensor_dataloader` to wrap training and selection arrays into device-staged, full-batch PyTorch `DataLoader` instances using `TORCH_DEVICE` and `TORCH_DTYPE`.

In [ ]:
def create_dataloaders(tr_x: np.ndarray, tr_y: np.ndarray, sl_x: np.ndarray, sl_y: np.ndarray):
    train_loader = make_tensor_dataloader(
        tr_x,
        tr_y,
        device=TORCH_DEVICE,
        dtype=TORCH_DTYPE,
        batch_size=None,
        shuffle=False,
        stage_on_device=True,
    )
    select_loader = make_tensor_dataloader(
        sl_x,
        sl_y,
        device=TORCH_DEVICE,
        dtype=TORCH_DTYPE,
        batch_size=None,
        shuffle=False,
        stage_on_device=True,
    )
    return train_loader, select_loader

### 5.2 GMDH Model Factory (`get_gmdh`)

The GMDH architecture is assembled from the following components:
- **`GMDHModel`**: A `torch.nn.Sequential` subclass holding the sparse layers, normalizations, activations, and output head.
- **`GMDHLayer` (Eager Layer 1 + Growth Template)**: Starts already materialized at Layer 1 (`in_features=input_features, init="xavier"`) so trainable parameters exist on Batch 1 of Epoch 1 without a closed-form solver. Subsequent layers use `forward=True`, `inputs=2, degree=1, interaction=2`, and `max_neurons=50`.
- **`GMDHNorm` (`norm=True`) & `SoftClip(3.0)`**: Per-neuron standardization and smooth bounded activation placed between layers (preserving pass-through for forwarded channels).
- **`GMDHOutput(count=..., dense=True)`**: Learnable linear readout head combining all active neurons of the final `GMDHLayer`, enabling gradients to flow through all hidden sparse neurons.
- **`ModelIncrementer` (`solver=None`)**: Orchestrates solver-free piecewise growth (`inc.grow`), lightly pruning weak neurons by `GMDHOutput` contribution (`SelectRelativeThreshold(0.01)`), warm-inserting new sparse layers (`resize_warm`), and applying `final_select` + `trim()` at the end.

In [ ]:
def get_gmdh(input_features: int, max_layers: int, output_features: int, logs: bool = True):
    layer1 = GMDHLayer(
        in_features=input_features,
        inputs=2,
        degree=1,
        interaction=2,
        init="xavier",
        dtype=TORCH_DTYPE,
        device=TORCH_DEVICE,
    )
    model = GMDHModel(
        layer1,
        GMDHOutput(
            count=output_features,
            dense=True,
            in_features=layer1.out_features,
            dtype=TORCH_DTYPE,
            device=TORCH_DEVICE,
        ),
        dtype=TORCH_DTYPE,
        device=TORCH_DEVICE,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    incrementer = ModelIncrementer(
        model,
        optimizer=optimizer,
        layer=GMDHLayer(
            inputs=2,
            degree=1,
            interaction=2,
            forward=True,
            max_neurons=50,
            init="xavier",
        ),
        activation=SoftClip(3.0, dtype=TORCH_DTYPE, device=TORCH_DEVICE),
        norm=True,
        select=SelectRelativeThreshold(0.01),
        final_select=SelectRelativeThreshold(0.05),
        max_layers=max_layers,
        gradient_during_growth=True,
        solver=None,
        logs=logs,
    )

    return model, optimizer, incrementer

### 5.3 Dense MLP Factory (`get_mlp`)

`ModelIncrementer` accepts any `nn.Module`: when passed a standard PyTorch MLP (containing no `GMDHModel`), it automatically operates in **pass-through no-op mode** (`growing == False`), allowing us to use the exact same training loop for both MLP and GMDH without any custom dummy wrapper class.

In `get_mlp`, we use fixed hidden layer widths (`6 -> 15 -> 10`) comparable in scale to the GMDH candidate layers.

In [ ]:
def get_mlp(input_features: int, output_features: int):
    model = nn.Sequential(
        nn.Linear(in_features=input_features, out_features=6, dtype=TORCH_DTYPE, device=TORCH_DEVICE),
        nn.ReLU(),
        nn.Linear(in_features=6, out_features=15, dtype=TORCH_DTYPE, device=TORCH_DEVICE),
        nn.ReLU(),
        nn.Linear(in_features=15, out_features=10, dtype=TORCH_DTYPE, device=TORCH_DEVICE),
        nn.ReLU(),
        nn.Linear(in_features=10, out_features=output_features, dtype=TORCH_DTYPE, device=TORCH_DEVICE),
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    return model, optimizer, ModelIncrementer(model, logs=False)

## 6. Unified Training Loop & Evaluation Functions

### How Piecewise Sparse GMDH Training Works (`train_model`)

Unlike a traditional Multi-Layer Perceptron (MLP)—where all layers exist from the start—a **sparse GMDH** network **grows its architecture piecewise during training**:

1. **Eager Sparse Initialization (Layer 1)**: `get_gmdh()` initializes Layer 1 eagerly (`in_features=input_features`, `init="xavier"`) followed by `GMDHOutput(dense=True)`. Trainable parameters exist from Batch 1 of Epoch 1, so backpropagation starts immediately alongside the MLP.
2. **Standard PyTorch Gradient Training (`every epoch`)**: Both MLP and GMDH run the exact same `optimizer.zero_grad()`, `loss.backward()`, and `optimizer.step()` pass.
3. **Piecewise Layer Growth (`inc.grow(...)`, every `grow_epochs`)**:
   - Scores top-layer neurons by their contribution through `GMDHOutput` (`||W[:, j]||_2 * RMS(H[:, j])`) on `select_loader`.
   - Lightly prunes weak neurons (`SelectRelativeThreshold(0.01)`), computes unlabeled `GMDHNorm` statistics from `train_loader`, and inserts a new sparse `GMDHLayer` (`forward=True`) + `SoftClip(3.0)`.
   - Calls `GMDHOutput.resize_warm()` to copy learned weights onto forwarded channels and zero-initialize new candidate columns (preventing loss spikes on growth).
   - Once `max_layers` is reached, applies `final_select` (`SelectRelativeThreshold(0.05)`) and `model.trim()` to collapse the network to its minimal sparse subgraph.
4. **Selection Evaluation & Live Progress Plot**: Evaluates MSE on `select_loader` and refreshes a full-width live chart.

In [ ]:
def train_model(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    inc: ModelIncrementer,
    train_loader,
    select_loader,
    total_epochs: int = 2200,
    title: str = "Model Training",
):
    loss_fn = nn.MSELoss()
    grow_epochs = max(1, total_epochs // 10)
    plot_epochs = max(1, total_epochs // 10)

    select_losses = np.zeros(total_epochs, dtype=np.float64)
    min_select_loss = float("inf")
    n_select = len(select_loader.dataset)

    t0 = time.perf_counter()
    fig, ax = plt.subplots(figsize=(15, 4))

    for epoch in range(1, total_epochs + 1):
        # Standard PyTorch gradient pass (works from Epoch 1, Batch 1 for both GMDH and MLP)
        model.train()
        for xb, yb in train_loader:
            if xb.device != TORCH_DEVICE:
                xb = xb.to(TORCH_DEVICE)
                yb = yb.to(TORCH_DEVICE)
            if yb.ndim == 1:
                yb = yb.unsqueeze(1)
            optimizer.zero_grad()
            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()

        # Piecewise GMDH growth every `grow_epochs` (no-op for MLP since `inc.growing` is False)
        if inc.growing and (epoch % grow_epochs == 0):
            inc.grow(select_loader=select_loader, train_loader=train_loader)

        # Selection evaluation pass & periodic live plot
        model.eval()
        sel_loss_accum = 0.0
        with torch.no_grad():
            for xb, yb in select_loader:
                if xb.device != TORCH_DEVICE:
                    xb = xb.to(TORCH_DEVICE)
                    yb = yb.to(TORCH_DEVICE)
                if yb.ndim == 1:
                    yb = yb.unsqueeze(1)
                y_pred = model(xb)
                sel_loss_accum += loss_fn(y_pred, yb).item() * len(xb)
        select_loss = sel_loss_accum / n_select
        min_select_loss = min(select_loss, min_select_loss)
        select_losses[epoch - 1] = select_loss

        if (epoch % plot_epochs == 0) or (epoch == total_epochs):
            clear_output(wait=True)
            ax.clear()
            ax.plot(range(1, epoch + 1), select_losses[:epoch], "-", label="Selection Loss (MSE)")
            ax.axhline(min_select_loss, linestyle="--", color="goldenrod", label=f"Min Sel. Loss ({min_select_loss:.4e})")
            ax.set_title(f"{title} — Epoch {epoch}/{total_epochs} (Train Loss: {loss.item():.4e})")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("MSE")
            ax.grid(True)
            ax.legend()
            fig.tight_layout()
            display(fig)

    plt.close(fig)
    train_time = time.perf_counter() - t0
    print(f"Training finished in {train_time:.2f} s | Minimum Selection Loss: {min_select_loss:.6e}")
    return min_select_loss, train_time, select_losses


def evaluate_single_target(
    model: nn.Module,
    vl_x: np.ndarray,
    vl_y: np.ndarray,
    min_select_loss: float,
    train_time: float,
    vl_pred_classes: np.ndarray | None = None,
):
    model.eval()
    if vl_pred_classes is None:
        with torch.no_grad():
            raw_out = model(torch.as_tensor(vl_x, dtype=TORCH_DTYPE, device=TORCH_DEVICE)).detach().cpu().numpy()
        vl_output = np.round(raw_out.ravel())
    else:
        vl_output = vl_pred_classes

    vl_nrmse = nrmse(vl_y, vl_output) * 100.0
    accuracy = float(np.mean(vl_output == vl_y) * 100.0)

    print("Trained model:")
    print(model)
    print("\nResults:")
    print(f"- training time (s)      : {train_time:6.2f}")
    print(f"- min. selection MSE     : {min_select_loss:8.5f}")
    print(f"- validation NRMSE (%)   : {vl_nrmse:8.5f}")
    print(f"- validation Accuracy (%): {accuracy:6.2f}%")

    plt.figure(figsize=(15, 4))
    plt.plot(vl_y, ".-", label="Actual Class")
    plt.plot(vl_output, "o--", label="Estimated Class", markerfacecolor="none")
    plt.xlabel("Validation Sample Index")
    plt.ylabel("Class ID")
    plt.yticks(list(classes.keys()), [f"{k} ({v})" for k, v in classes.items()])
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    return vl_nrmse

---

## 7. Experiment 1: Single-Output Classification ($y \in \{1, 2, 3\}$)

We now instantiate the single-target `DataLoader`s along with both the **Dense MLP** and **GMDH** models.

In [ ]:
train_loader, select_loader = create_dataloaders(tr_samples, tr_targets, sl_samples, sl_targets)

input_features = tr_samples.shape[1]

mlp_single, mlp_opt_single, mlp_inc_single = get_mlp(input_features=input_features, output_features=1)

gmdh_single, gmdh_opt_single, gmdh_inc_single = get_gmdh(input_features=input_features, max_layers=4, output_features=1, logs=True)

print("Single-target DataLoaders, MLP, and GMDH models initialized.")

### 7.1 Train & Validate Dense MLP (Single Output)

#### Train

In [ ]:
mlp_min_sel, mlp_time, _ = train_model(
    mlp_single,
    mlp_opt_single,
    mlp_inc_single,
    train_loader,
    select_loader,
    total_epochs=200,
    title="Dense MLP (Single Output)",
)

#### Validate

In [ ]:
mlp_single_nrmse = evaluate_single_target(
    mlp_single,
    vl_samples,
    vl_targets,
    mlp_min_sel,
    mlp_time,
)

### 7.2 Train & Validate GMDH Network (Single Output)

#### Train

In [ ]:
gmdh_min_sel, gmdh_time, _ = train_model(
    gmdh_single,
    gmdh_opt_single,
    gmdh_inc_single,
    train_loader,
    select_loader,
    total_epochs=600,
    title="GMDH Sparse Perceptron (Single Output)",
)

#### Validate

In [ ]:
gmdh_single_nrmse = evaluate_single_target(
    gmdh_single,
    vl_samples,
    vl_targets,
    gmdh_min_sel,
    gmdh_time,
)

---

## 8. Experiment 2: Multi-Output (One-Hot) Classification ($\mathbf{y} \in \{0, 1\}^3$)

Instead of regressing a single ordinal target $y \in \{1, 2, 3\}$ (which imposes an artificial metric ordering between classes), we now represent the 3 species as **3 binary indicator outputs** in $[0, 1]$, one column per class:
- Class 1 (*Iris Setosa*): $[1, 0, 0]$
- Class 2 (*Iris Versicolour*): $[0, 1, 0]$
- Class 3 (*Iris Virginica*): $[0, 0, 1]$

We define helper functions to convert single-column class IDs into multi-output indicator matrices (`to_multiple_targets`) and to convert multi-output predictions back into scalar class IDs (`from_multiple_targets`).

In [ ]:
def to_multiple_targets(single_targets: np.ndarray, num_classes: int = len(classes)) -> np.ndarray:
    multi = np.zeros((len(single_targets), num_classes), dtype=single_targets.dtype)
    for idx, cls_id in enumerate(single_targets.astype(int)):
        multi[idx, cls_id - 1] = 1.0
    return multi


def from_multiple_targets(multi_outputs: np.ndarray) -> np.ndarray:
    vl_output = np.zeros(len(multi_outputs), dtype=multi_outputs.dtype)
    for k in range(multi_outputs.shape[1]):
        vl_output[multi_outputs[:, k] > 0.5] = k + 1
    return vl_output

### Creating the `_multiple` Target Arrays and DataLoaders

In [ ]:
tr_targets_multiple = to_multiple_targets(tr_targets)
sl_targets_multiple = to_multiple_targets(sl_targets)
vl_targets_multiple = to_multiple_targets(vl_targets)

train_loader_multiple, select_loader_multiple = create_dataloaders(
    tr_samples, tr_targets_multiple, sl_samples, sl_targets_multiple
)

print(f"tr_targets_multiple shape: {tr_targets_multiple.shape}")
print(f"sl_targets_multiple shape: {sl_targets_multiple.shape}")
print(f"vl_targets_multiple shape: {vl_targets_multiple.shape}")

### 8.1 Train & Validate Dense MLP (Multiple Outputs)

#### Train

In [ ]:
output_features_multi = tr_targets_multiple.shape[1]
mlp_multi, mlp_opt_multi, mlp_inc_multi = get_mlp(
    input_features=input_features,
    output_features=output_features_multi,
)

mlp_multi_min_sel, mlp_multi_time, _ = train_model(
    mlp_multi,
    mlp_opt_multi,
    mlp_inc_multi,
    train_loader_multiple,
    select_loader_multiple,
    total_epochs=200,
    title="Dense MLP (Multiple Outputs)",
)

#### Validate

In [ ]:
mlp_multi.eval()
with torch.no_grad():
    mlp_multi_raw = mlp_multi(torch.as_tensor(vl_samples, dtype=TORCH_DTYPE, device=TORCH_DEVICE)).detach().cpu().numpy()

mlp_multi_pred_classes = from_multiple_targets(mlp_multi_raw)
mlp_multi_nrmse = evaluate_single_target(
    mlp_multi,
    vl_samples,
    vl_targets,
    mlp_multi_min_sel,
    mlp_multi_time,
    vl_pred_classes=mlp_multi_pred_classes,
)

### 8.2 Train & Validate GMDH Network (Multiple Outputs)

#### Train

In [ ]:
gmdh_multi, gmdh_opt_multi, gmdh_inc_multi = get_gmdh(
    input_features=input_features,
    max_layers=4,
    output_features=output_features_multi,
    logs=True,
)

gmdh_multi_min_sel, gmdh_multi_time, _ = train_model(
    gmdh_multi,
    gmdh_opt_multi,
    gmdh_inc_multi,
    train_loader_multiple,
    select_loader_multiple,
    total_epochs=450,
    title="GMDH Sparse Perceptron (Multiple Outputs)",
)

#### Validate

In [ ]:
gmdh_multi.eval()
with torch.no_grad():
    gmdh_multi_raw = gmdh_multi(torch.as_tensor(vl_samples, dtype=TORCH_DTYPE, device=TORCH_DEVICE)).detach().cpu().numpy()

gmdh_multi_pred_classes = from_multiple_targets(gmdh_multi_raw)
gmdh_multi_nrmse = evaluate_single_target(
    gmdh_multi,
    vl_samples,
    vl_targets,
    gmdh_multi_min_sel,
    gmdh_multi_time,
    vl_pred_classes=gmdh_multi_pred_classes,
)